# Ablation A — MLP-LoRA E1 WSFT Inference (1.5B + 3B)

Runs inference on the 100 test questions for the MLP-LoRA E1 adapters
trained in `Ablation_A_E1_MLP_LoRA_Train.ipynb`.

**Prereq:** Trained adapters in `runs/e1_mlp_lora/{model}/`

**Output:** `data/e1_mlp_lora_inference_100_TESTONLY.json`

This runs on the local GPU (Windows) — start it and let it run while
the judge notebook runs in parallel via Vertex AI on RAG.

In [1]:
# Cell 0: Config + load questions
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["CUDA_DEVICE_MAX_CONNECTIONS"] = "1"

import json, time, gc
import torch

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"
# PROJECT_DIR = os.path.expanduser("~/kd_project")  # Linux

DATA_DIR = os.path.join(PROJECT_DIR, "data")
RUNS_DIR = os.path.join(PROJECT_DIR, "runs")

N_EVAL = 100
GEN_KW = dict(max_new_tokens=2000, do_sample=False)

QUESTIONS_FILE = os.path.join(DATA_DIR, "eval_questions_100.json")
with open(QUESTIONS_FILE) as f:
    q_data = json.load(f)
eval_questions = q_data["questions"][:N_EVAL]

# Only 1.5B + 3B (skip 7B for time — paper ablation only needs to show direction)
STUDENT_PATHS = {
    "qwen25_1p5b_base": "Qwen/Qwen2.5-1.5B",
    "qwen25_3b_base":   "Qwen/Qwen2.5-3B",
}

MLP_LORA_DIR = os.path.join(RUNS_DIR, "e1_mlp_lora")
OUT_FILE = os.path.join(DATA_DIR, f"e1_mlp_lora_inference_{N_EVAL}_TESTONLY.json")

print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'NONE'}")
print(f"Questions: {len(eval_questions)}")
print(f"Output: {OUT_FILE}")

# Verify adapters exist
for sname in STUDENT_PATHS:
    adapter = os.path.join(MLP_LORA_DIR, sname)
    has = os.path.exists(os.path.join(adapter, "adapter_config.json"))
    status = "✅" if has else "❌ NOT FOUND — train it first"
    print(f"  {status} {sname}: {adapter}")

GPU: NVIDIA GeForce RTX 4090
Questions: 100
Output: C:\Users\adishalit1\Desktop\kd_project\data\e1_mlp_lora_inference_100_TESTONLY.json
  ✅ qwen25_1p5b_base: C:\Users\adishalit1\Desktop\kd_project\runs\e1_mlp_lora\qwen25_1p5b_base
  ✅ qwen25_3b_base: C:\Users\adishalit1\Desktop\kd_project\runs\e1_mlp_lora\qwen25_3b_base


In [2]:
# Cell 1: Run inference for each model size
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Resume support
if os.path.exists(OUT_FILE):
    with open(OUT_FILE) as f:
        result = json.load(f)
    done_models = set(result.get("models", {}).keys())
    print(f"Resume: {done_models} already done")
else:
    result = {
        "meta": {"source": "eval_questions_100.json", "n_samples": N_EVAL,
                 "method": "E1 WSFT + MLP-LoRA (FFN target_modules, r=32)"},
        "models": {},
        "samples": [{"id": q["id"], "prompt": q["prompt"], "outputs": {}} for q in eval_questions],
    }
    done_models = set()

@torch.inference_mode()
def run_inference(model, tokenizer):
    answers = []
    for sample in result["samples"]:
        inputs = tokenizer(sample["prompt"], return_tensors="pt", truncation=True)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
        out = model.generate(**inputs, **GEN_KW)
        decoded = tokenizer.decode(out[0], skip_special_tokens=True)
        answer = decoded[len(sample["prompt"]):].lstrip() if decoded.startswith(sample["prompt"]) else decoded.strip()
        answers.append(answer)
    return answers

for sname, model_path in STUDENT_PATHS.items():
    if sname in done_models:
        print(f"\n⏩ {sname} already done")
        continue

    adapter = os.path.join(MLP_LORA_DIR, sname)
    if not os.path.exists(os.path.join(adapter, "adapter_config.json")):
        print(f"\n⚠️ {sname}: MLP-LoRA adapter not found at {adapter}")
        continue

    print(f"\n{'='*60}")
    print(f"  Loading {sname} + MLP-LoRA E1 adapter")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16,
        device_map="auto", trust_remote_code=True)

    model = PeftModel.from_pretrained(base, adapter)
    model.eval()

    t0 = time.time()
    answers = run_inference(model, tokenizer)
    elapsed = time.time() - t0
    print(f"  ✅ {len(answers)} samples in {elapsed/60:.1f} min ({elapsed/len(answers):.1f}s each)")

    result["models"][sname] = {"adapter": adapter, "path": model_path, "mlp_lora": True}
    for sample, answer in zip(result["samples"], answers):
        sample["outputs"][sname] = {"answer": answer}

    with open(OUT_FILE, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"  Saved → {OUT_FILE}")

    del model, base, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ All MLP-LoRA inference done")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  Loading qwen25_1p5b_base + MLP-LoRA E1 adapter


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 155.55it/s]
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Settin

  ✅ 100 samples in 52.2 min (31.3s each)
  Saved → C:\Users\adishalit1\Desktop\kd_project\data\e1_mlp_lora_inference_100_TESTONLY.json

  Loading qwen25_3b_base + MLP-LoRA E1 adapter


Loading weights: 100%|██████████| 434/434 [00:04<00:00, 100.69it/s]
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open

  ✅ 100 samples in 49.3 min (29.6s each)
  Saved → C:\Users\adishalit1\Desktop\kd_project\data\e1_mlp_lora_inference_100_TESTONLY.json

✅ All MLP-LoRA inference done
